# Routing Navigator — Transformer + REINFORCE Training

Trains the routing transformer with pointer-network attention and REINFORCE,
then distills to a lightweight student MLP for Tier-1 inference latency (<100 ms).

**Reward (independent — I-2)**: `-1 * (route_distance + tardiness_penalty + gini_unfairness_penalty)`
**Outputs**: MLflow run with `route_distance_mean`, `tardiness_p95`, `student_kd_loss`.

In [ ]:
%pip install --quiet torch torch-geometric mlflow gymnasium ray[rllib] structlog

In [ ]:
from __future__ import annotations
import os, random, numpy as np, torch, mlflow
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device={DEVICE}')

In [ ]:
mlflow.set_tracking_uri(os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5000'))
CITY = os.environ.get('CITY', 'bengaluru')
EXPERIMENT = f'{CITY}_routing_navigator' if CITY == 'bengaluru' else f'mumbai_transfer_routing_navigator'
mlflow.set_experiment(EXPERIMENT)

In [ ]:
# Build env (uses agents/routing_navigator)
import sys; sys.path.append('.')
from agents.routing_navigator.training.env import RoutingEnv
env = RoutingEnv(city=CITY, num_orders=20, seed=SEED)
obs, info = env.reset(seed=SEED)
print(env.observation_space, env.action_space)

In [ ]:
# Train teacher transformer with REINFORCE
from agents.routing_navigator.training.train import train_reinforce
with mlflow.start_run() as run:
    teacher, metrics = train_reinforce(env, episodes=100, lr=3e-4, seed=SEED, device=DEVICE)
    mlflow.log_metrics(metrics)
    print(metrics)

In [ ]:
# Distill to student MLP for Tier-1 inference
from agents.routing_navigator.training.distill import distill_to_student
with mlflow.start_run(nested=True) as student_run:
    student, kd_metrics = distill_to_student(teacher, env, epochs=20, device=DEVICE)
    mlflow.log_metrics(kd_metrics)
    mlflow.pytorch.log_model(student, artifact_path='student',
        registered_model_name=f'{"mumbai_" if CITY=="mumbai" else ""}routing_navigator_student')